# 09 · The last three context policies, and why kNN contexts fail (W1)

**Part A: three more policies from the proposal.** The balanced policy showed that putting every fraud in the context is what helps most. These three keep exactly the same frauds as balanced and change only *which legitimate rows* fill the rest, so the comparison with balanced isolates that choice:

| Policy | Legitimate rows in the context |
| --- | --- |
| cluster | spread over 50 k-means clusters of the legitimate rows, one from each cluster in turn |
| pca | spread along the 3 leading principal components of the fraud rows (5 × 5 × 5 cells, taken in turn) |
| hard_negative | half the legitimate rows closest to a fraud, half random ones, alternating |

TabICLv2 at 1% and 0.172%, k = 1K-10K, 5 seeds, plus XGBoost on the same rows so each policy has a classical counterpart.

**Part B: why kNN contexts fail.** `checks.knn_diagnosis` rebuilds the exact groups of every saved kNN run and measures two possible causes: groups whose context had no fraud (their rows all score 0), and scores on different scales across groups (good ranking inside each group, broken ranking when all groups are pooled).

**Runtime (T4 GPU):** Part A about 1.5 hours, Part B a few minutes on CPU. Runs are saved as they finish and skipped on a rerun.

In [ ]:
# 1. What to run.
SEEDS = [0, 1, 2, 3, 4]
LEVELS = ["natural", "1"]
NEW_POLICIES = ["cluster", "pca", "hard_negative"]
RUN_PART_A, RUN_PART_B = True, True

In [ ]:
# 2. Drive, latest code, libraries. Needs a GPU runtime for Part A.
from google.colab import drive
drive.mount("/content/drive")
import os, sys
REPO_URL = "https://github.com/Pakkabhabad18/tabular-fm-fraud-detection.git"
REPO_DIR = "/content/tabular-fm-fraud-detection"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
!pip install -q -r {REPO_DIR}/requirements.txt
sys.path.insert(0, f"{REPO_DIR}/src")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# 3. Tests, data, and the split fingerprint check.
!cd {REPO_DIR} && PYTHONPATH=src python -m pytest tests -q
from fraudicl.config import DRIVE_DIR, CONTEXT_BUDGETS
from fraudicl.data import load_ulb, features_and_target, load_splits, fingerprint
X, y = features_and_target(load_ulb())
splits = load_splits(f"{DRIVE_DIR}/splits/ulb_splits.npz")
expected = open(f"{REPO_DIR}/results/ulb_splits_fingerprint.txt").readline().strip()
assert fingerprint(splits) == expected, "splits on Drive differ from the ones on record"
OUT = f"{DRIVE_DIR}/results"
print("splits OK", expected)

In [ ]:
# 4. Part A: the three policies for TabICLv2, then XGBoost on the same rows.
from fraudicl.models import make_tabicl_v2
from fraudicl.experiments import run_policy_sweep, run_matched_xgb
if RUN_PART_A:
    run_policy_sweep(X, y, splits, OUT, {"TabICLv2": make_tabicl_v2}, NEW_POLICIES,
                     levels=LEVELS, budgets=CONTEXT_BUDGETS, seeds=SEEDS)
    for policy in NEW_POLICIES:
        run_matched_xgb(X, y, splits, OUT, levels=LEVELS, budgets=CONTEXT_BUDGETS,
                        seeds=SEEDS, policy=policy, variant="plain", n_trials=50)

In [ ]:
# 5. Part A: no context may contain a validation or test row, then the W1 table
#    with all ten policies, and each policy's TabICLv2 vs XGBoost on the same rows.
from fraudicl import report as R
from fraudicl.checks import context_leakage
leak = context_leakage(OUT, splits)
new = leak[leak.policy.isin(NEW_POLICIES)]
assert len(new) and (new[["outside_train", "in_validation", "in_test"]] == 0).all().all()
print(f"{len(new)} new contexts checked, none contains a validation or test row")

runs = R.load_runs(OUT)
policies = R.policy_table(runs, "TabICLv2", levels=LEVELS)
same_rows = (runs[runs.policy.isin(["balanced"] + NEW_POLICIES) & runs.level.astype(str).isin(LEVELS)]
             .groupby(["policy", "level", "k", "model"], observed=True).pr_auc.mean()
             .unstack("model").reset_index())
display(same_rows.round(4))
policies.round(4)

**How to read Part A.** In the policy table, compare each new policy with `balanced`, not only with `uniform`: all four hold the same frauds, so a difference from balanced is caused by the choice of legitimate rows alone. The second table puts TabICLv2 next to XGBoost trained on the same rows; if both move together, the gain comes from the rows, not from in-context learning.

In [ ]:
# 6. Part B: kNN diagnosis on every saved kNN run (CPU is enough).
import pandas as pd
from fraudicl.checks import knn_diagnosis
diag = []
if RUN_PART_B:
    by_key = {(s.seed, s.level): s for s in splits}
    for policy in ["knn", "knn_balanced"]:
        for level in LEVELS:
            for k in CONTEXT_BUDGETS:
                for seed in SEEDS:
                    d = knn_diagnosis(OUT, X, y, by_key[seed, level], policy, k)
                    if d:
                        diag.append(d)
diag = pd.DataFrame(diag)
diag_summary = (diag.groupby(["policy", "level", "k"])
                [["pooled_pr_auc", "pooled_roc_auc", "mean_within_group_roc_auc",
                  "share_test_rows_no_fraud_context", "share_frauds_scored_zero",
                  "legit_p99_ratio", "top_100_groups"]].mean().reset_index())
diag_summary.round(3)

**How to read Part B.** For `knn`, a large `share_frauds_scored_zero` means many frauds landed in groups whose context had no fraud at all and were given score 0: the policy fails because most groups never see a fraud. For `knn_balanced` every group sees all the frauds, so if PR-AUC is still low while `mean_within_group_roc_auc` is high, each group is ranked well on its own but the groups' scores are on different scales (`legit_p99_ratio` far above 1): legitimate rows from one group outrank frauds from another once all groups are pooled.

In [ ]:
# 7. Export: print as CSV text (copied into results/ on the laptop) and save on Drive.
EXPORT = f"{OUT}/export_09"
os.makedirs(EXPORT, exist_ok=True)
policies.to_csv(f"{EXPORT}/policy_table_tabiclv2.csv", index=False)
same_rows.to_csv(f"{EXPORT}/policy_same_rows_xgb.csv", index=False)
diag.to_csv(f"{EXPORT}/knn_diagnosis_runs.csv", index=False)
diag_summary.to_csv(f"{EXPORT}/knn_diagnosis_summary.csv", index=False)
print("=====POLICIES====="); print(policies.to_csv(index=False))
print("=====SAMEROWS====="); print(same_rows.to_csv(index=False))
print("=====KNN====="); print(diag_summary.to_csv(index=False))